## 1. AES 암호화 프로그램
-----

In [ ]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.padding import PKCS7
from cryptography.hazmat.backends import default_backend
import base64

# AES 암호화 함수
def aes_encrypt(plaintext, key, iv):
    # 키 길이를 16바이트로 맞추기 위해 패딩 또는 해시 처리
    key = key.ljust(16, '0').encode()[:16]  # 키를 16바이트로 패딩
    iv = iv.encode()[:16]  # IV를 16바이트로 맞춤

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()

    # PKCS7 패딩
    padder = PKCS7(algorithms.AES.block_size).padder()
    padded_data = padder.update(plaintext.encode()) + padder.finalize()

    # 암호화
    ciphertext = encryptor.update(padded_data) + encryptor.finalize()
    return base64.b64encode(ciphertext).decode()

# AES 복호화 함수
def aes_decrypt(ciphertext, key, iv):
    key = key.ljust(16, '0').encode()[:16]  # 키를 16바이트로 패딩
    iv = iv.encode()[:16]  # IV를 16바이트로 맞춤

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()

    # Base64 디코딩 및 복호화
    ciphertext = base64.b64decode(ciphertext)
    decrypted_data = decryptor.update(ciphertext) + decryptor.finalize()

    # PKCS7 패딩 제거
    unpadder = PKCS7(algorithms.AES.block_size).unpadder()
    plaintext = unpadder.update(decrypted_data) + unpadder.finalize()
    return plaintext.decode()

# 사용자 입력 기반 실행
if __name__ == "__main__":
    key = input("키를 입력하세요: ")  # 사용자 키 입력
    iv = "1234567890123456"  # IV는 고정값 사용 (필요시 사용자 입력으로 변경 가능)

    mode = input("모드를 선택하세요 (1: 암호화, 2: 복호화): ")

    if mode == "1":  # 암호화 모드
        plaintext = input("암호화할 문장을 입력하세요: ")
        encrypted = aes_encrypt(plaintext, key, iv)
        print(f"암호화 결과: {encrypted}")

    elif mode == "2":  # 복호화 모드
        ciphertext = input("복호화할 암호화된 문장을 입력하세요: ")
        try:
            decrypted = aes_decrypt(ciphertext, key, iv)
            print(f"복호화 결과: {decrypted}")
        except Exception as e:
            print(f"복호화 실패: {e}")

    else:
        print("잘못된 모드 선택입니다.")

키를 입력하세요: abcde
모드를 선택하세요 (1: 암호화, 2: 복호화): 1
암호화할 문장을 입력하세요: good
암호화 결과: mY/rShbTBdfE5MiRGPD0Ug==


## 2. 해시를 이용한 파일 변조 확인
-----

In [ ]:
import hashlib

# MD5 해시 계산 함수
def calculate_md5(file_path):
    md5 = hashlib.md5()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b""):
            md5.update(chunk)
    return md5.hexdigest()

# 파일 해시 비교 함수
def compare_files(file1_path, file2_path, file1_name, file2_name):
    # 해시값 계산
    file1_hash = calculate_md5(file1_path)
    file2_hash = calculate_md5(file2_path)

    # 해시값 출력
    print(f"파일 1: {file1_name}")
    print(f"MD5 해시값: {file1_hash}")
    print("-" * 40)

    print(f"파일 2: {file2_name}")
    print(f"MD5 해시값: {file2_hash}")
    print("-" * 40)

    # 비교 결과
    if file1_hash == file2_hash:
        print("결과: 두 파일은 동일합니다.")
    else:
        print("결과: 두 파일은 다른 파일입니다.")

# 주요 실행 코드
file1_path = '/content/dict_test_utf8-1.TXT'
file2_path = '/content/dict_test_utf8-2.TXT'
file1_name = "dict_test_utf8-1.TXT"
file2_name = "dict_test_utf8-2.TXT"

# 파일 비교 실행
compare_files(file1_path, file2_path, file1_name, file2_name)


파일 1: dict_test_utf8-1.TXT
MD5 해시값: 8098dbc08edb6b03cb27bdbe46a940c5
----------------------------------------
파일 2: dict_test_utf8-2.TXT
MD5 해시값: 6216be52244395876887469e48efe64e
----------------------------------------
결과: 두 파일은 다른 파일입니다.
